# Arabic Cursive OCR Benchmark — Hosted APIs on Colab

Same sweep engine as `04_colab_sweep.ipynb` (partial then full phase,
per-phase checkpoint + push, resume-safe) — but the models are hosted
APIs, so **no GPU**. Add whichever Colab secrets you have keys for:
`ANTHROPIC_API_KEY`, `GEMINI_API_KEY` (free tier at aistudio.google.com/apikey),
`OPENAI_API_KEY`. Only models with a key present are run.

## 1. Repo

In [ ]:
# Bulletproof checkout: works whether the repo is missing, at the wrong
# path, or has local edits. Always ends up at the latest origin/main.
import os
REPO = '/content/arabic-ocr-benchmark'
token = None
try:
    from google.colab import userdata
    token = userdata.get('GH_TOKEN')
except Exception as e:
    print('no GH_TOKEN secret:', e)
url = f"https://{(token + '@') if token else ''}github.com/rabeeh97/arabic-ocr-benchmark.git"
if os.path.isdir(REPO + '/.git'):
    os.system(f'git -C {REPO} fetch --depth 1 origin main && git -C {REPO} reset --hard origin/main')
    # drop untracked leftovers (e.g. a stale sweep_manifest.json) so the
    # manifest is always exactly what's in the repo. Keeps gitignored _cache.
    os.system(f'git -C {REPO} clean -fd -- results')
else:
    os.system(f'rm -rf {REPO} && git clone --depth 1 {url} {REPO}')
os.chdir(REPO)
# src-layout package: importable straight from the checkout even if the
# `pip install -e` cell fails or is skipped (this is what caused the old
# 'No module named arabic_ocr_bench' in the sweep cell).
import sys
for _p in (os.path.join(os.getcwd(), 'src'), os.getcwd()):
    if _p not in sys.path: sys.path.insert(0, _p)
print(os.popen('git log --oneline -3').read())

## 2. Install + caches

In [ ]:
# Keep all downloads on Colab's own scratch disk (ephemeral, ~78 GB) --
# never the local machine. Nothing here touches your computer.
import os
os.environ['HF_HOME'] = '/content/hf_cache'
os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
# Optional HF_TOKEN secret: lifts anonymous rate limits and speeds the
# Muharaf download (and is required for gated models like PaliGemma's base).
try:
    from google.colab import userdata
    _hf = userdata.get('HF_TOKEN')
    if _hf:
        os.environ['HF_TOKEN'] = _hf
        print('HF_TOKEN set')
except Exception:
    pass

In [ ]:
!pip install -q -e '.'   # core only — no torch/transformers

## 3. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 4. Keys -> models

Copies whatever provider secrets you've added into the environment, then
`available_hosted_models()` selects **every `client: litellm` model in
`configs/models.yaml` whose key is present** — so adding a model there is
all it takes. Add these Colab secrets as needed: `ANTHROPIC_API_KEY`,
`GEMINI_API_KEY`, `OPENAI_API_KEY`, `MISTRAL_API_KEY`, `OPENROUTER_API_KEY`,
`OLLAMA_API_KEY` + `OLLAMA_API_BASE`.

In [ ]:
import os
from google.colab import userdata
for k in ('ANTHROPIC_API_KEY', 'GEMINI_API_KEY', 'OPENAI_API_KEY',
          'MISTRAL_API_KEY', 'OPENROUTER_API_KEY',
          'OLLAMA_API_KEY', 'OLLAMA_API_BASE', 'HF_TOKEN'):
    try:
        v = userdata.get(k)
        if v: os.environ[k] = v
    except Exception:
        pass

from arabic_ocr_bench.models.registry import available_hosted_models
MODELS = available_hosted_models()
assert MODELS, 'no provider API-key secrets found — add at least one'
print('will run:', MODELS)

# KITAB-Bench subsets are small + ungated: kitab_khatt (handwriting),
# kitab_muharaf, kitab_historicalbooks, kitab_hindawi. Or 'muharaf_lines'.
DATASETS = ['kitab_khatt']   # add more, e.g. ['kitab_khatt', 'kitab_hindawi']
PARTIAL_LIMIT = 20       # paid APIs — keep the partial small
RUN_FULL = False         # KITAB subsets are only 200-500 rows; True does the whole split
ACCUMULATE = True        # each re-run adds PARTIAL_LIMIT NEW lines/model + retries errors;
                         # nothing already scored is recomputed. Set False for the old
                         # 'run each model once, skip if done' behaviour.

## 5. Run the sweep — checkpoints + pushes after every model/phase

In [ ]:
import os, sys, shutil, subprocess, pathlib
# make the src-layout package importable even if `pip install -e` didn't take
for _p in (os.path.join(os.getcwd(), 'src'), os.getcwd()):
    if _p not in sys.path: sys.path.insert(0, _p)
from arabic_ocr_bench.sweep import run_sweep
from arabic_ocr_bench.samples import export_run_samples

DRIVE = pathlib.Path('/content/drive/MyDrive/arabic-ocr-bench')
DRIVE_OK = DRIVE.parent.exists()   # is Drive mounted?

_DS = {}
_HF_SPLIT = {
    'muharaf_lines': ('aamijar/muharaf-public', 'test'),
    'kitab_khatt': ('ahmedheakl/arocrbench_khatt', 'train'),
    'kitab_muharaf': ('ahmedheakl/arocrbench_muharaf', 'train'),
    'kitab_historicalbooks': ('ahmedheakl/arocrbench_historicalbooks', 'train'),
    'kitab_hindawi': ('ahmedheakl/arocrbench_hindawi', 'train'),
}  # add datasets here
def _get_image_fn(dataset):
    # row-index -> PIL image, for the sample export
    if dataset not in _HF_SPLIT:
        return lambda i: None   # unknown dataset -> samples show 'image unavailable'
    if dataset not in _DS:
        from datasets import load_dataset
        hf_id, split = _HF_SPLIT[dataset]
        _DS[dataset] = load_dataset(hf_id, split=split)
    ds = _DS[dataset]
    return lambda i: ds[int(i)]['image']

def _git(*a):
    return subprocess.run(['git', *a], capture_output=True, text=True)

def _mirror_to_drive(run_dir):
    if not DRIVE_OK:
        return
    (DRIVE / 'results').mkdir(parents=True, exist_ok=True)
    if pathlib.Path('results/_cache').is_dir():
        shutil.copytree('results/_cache', DRIVE / '_cache', dirs_exist_ok=True)
    shutil.copytree(run_dir, DRIVE / 'results' / pathlib.Path(run_dir).name, dirs_exist_ok=True)
    for f in ('sweep_manifest.json', 'leaderboard.html', 'leaderboard.json'):
        if pathlib.Path('results', f).is_file():
            shutil.copy2(pathlib.Path('results', f), DRIVE / 'results' / f)

def checkpoint(entry, run_dir):
    # Called by run_sweep AFTER a phase is scored. Persist the results here.
    # RAISE if they don't make it to the repo -> the phase is rolled back and
    # retried next run (so a 'done' entry always means 'in the repo').
    run_dir = str(run_dir)
    try:
        export_run_samples(run_dir, get_image=_get_image_fn(entry['dataset']), k=3)
    except Exception as e:
        print('   sample export failed:', e)
    subprocess.run(['python', 'scripts/make_leaderboard.py', 'results',
                    '-o', 'results/leaderboard.html'])
    _mirror_to_drive(run_dir)
    if not token:
        print('   no GH_TOKEN -> Drive only (this counts as persisted)')
        return
    _git('config', 'user.email', 'colab@arabic-ocr-bench')
    _git('config', 'user.name', 'colab-sweep')
    _git('add', '-f', run_dir, 'results/_cache', 'results/sweep_manifest.json',
         'results/leaderboard.html', 'results/leaderboard.json')
    _git('commit', '-m', f"sweep: {entry['model']} {entry['phase']} "
         f"CER-light {entry['cer_micro_light']*100:.1f}% n={entry['n']}")
    _git('pull', '--rebase', '--autostash', 'origin', 'main')
    p = _git('push', 'origin', 'HEAD:main')
    if p.returncode != 0:
        raise RuntimeError('git push failed: ' + (p.stderr.strip()[:300] or 'unknown'))
    print('   push: ok')

# The manifest is git-tracked (comes from the fresh checkout) — it is the
# single source of truth for what's done. Only restore the per-line cache
# from Drive, to make a re-running phase fast.
if DRIVE_OK and (DRIVE / '_cache').is_dir():
    shutil.copytree(DRIVE / '_cache', 'results/_cache', dirs_exist_ok=True)
    print('restored line cache from Drive')

# One run_sweep per dataset. Each is independently resumable and wrapped so
# a dataset that fails outright (e.g. a gated download) doesn't stop the rest.
# Per-model / per-phase / per-line failures are already caught inside run_sweep.
for _ds in DATASETS:
    print(f'\n########## DATASET: {_ds} ##########')
    try:
        run_sweep(
            models=MODELS, dataset=_ds,
            partial_limit=PARTIAL_LIMIT, run_full=RUN_FULL,
            accumulate=ACCUMULATE,
            concurrency=4,
            manifest_path='results/sweep_manifest.json', results_dir='results',
            on_progress=print, on_done=checkpoint,
        )
    except Exception as e:
        import traceback; traceback.print_exc()
        print(f'!!! DATASET {_ds} FAILED ({type(e).__name__}: {e}) — continuing !!!')
print('\n=== SWEEP COMPLETE ===')

## 6. Final leaderboard

In [ ]:
!python scripts/make_leaderboard.py results -o results/leaderboard.html
from IPython.display import HTML
HTML(open('results/leaderboard.html').read())

## Notes

- `cost_usd_total` in the summary is estimated from `configs/models.yaml` pricing.
- Output is capped at 256 tokens/line (a line is <=181 chars) — same idea as the local 160-token cap.
- Gemini's free tier is rate-limited; the runner retries 429s with backoff.
- Refusals are detected (English + Arabic) and scored as empty, not as the apology text.
- A line that comes back with a transport/API error is **not scored, not cached** — the next run retries exactly those lines.
- `ACCUMULATE = True` (default here): every re-run adds `PARTIAL_LIMIT` brand-new lines per model and retries past errors; nothing already scored is recomputed. `ACCUMULATE = False` falls back to the old skip-if-in-manifest behaviour.